In [1]:
import numpy as np 
import pandas as pd 
from tqdm import tqdm

In [2]:
round_slots = pd.read_csv('data/MNCAATourneySlots.csv')
round_slots = round_slots[round_slots['Season'] == 2023]
round_slots = round_slots[round_slots['Slot'].str.contains('R')] # Filter out First Four

seeds = pd.read_csv('data/2024_tourney_seeds.csv')
seeds_m = seeds[seeds['Tournament'] == 'M']
seeds_w = seeds[seeds['Tournament'] == 'W']

# Predictions of last year's 1st place solution by RustyB: https://www.kaggle.com/code/rustyb/paris-madness-2023/output
preds = pd.read_csv('data/bracket_predictions.csv') 
preds['ID'] = preds['ID'].str.split('_')
preds['Pred'] = preds['pred_prob']

In [11]:
seeds_m[seeds_m['Seed']=="X16"]['TeamID'] <- 1447

31    False
Name: TeamID, dtype: bool

In [12]:
def prepare_data(seeds, preds):
    # Function preparing the data for the simulation
    seed_dict = seeds.set_index('Seed')['TeamID'].to_dict()
    inverted_seed_dict = {value: key for key, value in seed_dict.items()}
    probas_dict = {}
    
    for teams, proba in zip(preds['ID'], preds['Pred']):
        team1, team2 = teams[1], teams[2]

        probas_dict.setdefault(team1, {})[team2] = proba
        probas_dict.setdefault(team2, {})[team1] = 1 - proba

    return seed_dict, inverted_seed_dict, probas_dict

In [13]:
def simulate(round_slots, seeds, inverted_seeds, probas, random_values, sim=True):
    '''
    Simulates each round of the tournament.

    Parameters:
    - round_slots: DataFrame containing information on who is playing in each round.
    - seeds (dict): Dictionary mapping seed values to team IDs.
    - inverted_seeds (dict): Dictionary mapping team IDs to seed values.
    - probas (dict): Dictionary containing matchup probabilities.
    - random_values (array-like): Array with precomputed random-values.
    - sim (boolean): Simulates match if True. Chooses team with higher probability as winner otherwise.

    Returns:
    - list: List with winning team IDs for each match.
    - list: List with corresponding slot names for each match.
    '''
    winners = []
    slots = []

    for slot, strong, weak, random_val in zip(round_slots.Slot, round_slots.StrongSeed, round_slots.WeakSeed, random_values):
        team1, team2 = seeds[strong], seeds[weak]

        # Get the probability of team_1 winning
        proba = probas[str(team1)][str(team2)]
            
        if sim:
            # Randomly determine the winner based on the probability
            winner = team1 if random_val < proba else team2
        else:
            # Determine the winner based on the higher probability
            winner = [team1, team2][np.argmax([proba, 1-proba])]
            
        # Append the winner and corresponding slot to the lists
        winners.append(winner)
        slots.append(slot)

        seeds[slot] = winner

    # Convert winners to original seeds using the inverted_seeds dictionary
    return [inverted_seeds[w] for w in winners], slots


def run_simulation(brackets=1, seeds=None, preds=None, round_slots=None, sim=True):
    '''
    Runs a simulation of bracket tournaments.

    Parameters:
    - brackets (int): Number of brackets to simulate.
    - seeds (pd.DataFrame): DataFrame containing seed information.
    - preds (pd.DataFrame): DataFrame containing prediction information for each match-up.
    - round_slots (pd.DataFrame): DataFrame containing information about the tournament rounds.
    - sim (boolean): Simulates matches if True. Chooses team with higher probability as winner otherwise.

    Returns:
    - pd.DataFrame: DataFrame with simulation results.
    '''
    # Get relevant data for the simulation
    seed_dict, inverted_seed_dict, probas_dict = prepare_data(seeds, preds)
    # Lists to store simulation results
    results = []
    bracket = []
    slots = []
    
    # Precompute random-values
    random_values = np.random.random(size=(brackets, len(round_slots)))

    # Iterate through the specified number of brackets
    for b in tqdm(range(1, brackets+1)):
        # Run single simulation
        r, s = simulate(round_slots, seed_dict, inverted_seed_dict, probas_dict, random_values[b-1], sim)
        
        # Update results
        results.extend(r)
        bracket.extend([b] * len(r))
        slots.extend(s)

    # Create final DataFrame
    result_df = pd.DataFrame({'Bracket': bracket, 'Slot': slots, 'Team': results})

    return result_df

In [14]:
n_brackets = 100000
result_m=run_simulation(brackets=n_brackets, seeds=seeds_m, preds=preds, round_slots=round_slots, sim=True)

100%|██████████| 100000/100000 [00:04<00:00, 24228.20it/s]


In [15]:
result_m.to_csv('bracket_sims.csv',index=False)

In [16]:
def make_implied_probability_table(df_sub):
    # pandas gibberish that gets you the proportion of times Team wins
    # a particular slot in the tournament
    tmp = df_sub[['Tournament','Slot','Team']]\
            .groupby(['Tournament','Slot'])\
            .agg('value_counts',normalize=True)
    
    # more pandas gibberish to get it in the format we want.
    # eventually want the columns to be named after rounds and have
    # rows correspond to tournament and team
    tmp = tmp.to_frame()
    tmp.reset_index(inplace=True)
    tmp['Round'] = tmp['Slot'].str[0:2]
    tmp.drop(columns='Slot', inplace=True)
    tmp.set_index(['Tournament','Team','Round'], inplace=True)
    tmp = tmp.stack().unstack(level=2).fillna(0.0)
    tmp.reset_index(inplace=True)
    
    # cleanup
    tmp.columns.name=None
    tmp.drop(columns='level_2',inplace=True)
    
    # now need to add in missing teams, if any
    # some teams may never appear in the bracket.  This means they
    # should have implied probabilities of 0 for all rounds
    df_missing = []
    seeds = [f'{region}{num:02d}' for region in list('WXYZ') \
                 for num in range(1,17)]
    for t, sdf in tmp.groupby('Tournament'):
        missing_seeds = np.setdiff1d(seeds, sdf['Team'])
        df_missing.append(pd.DataFrame({'Tournament':t, 'Team': missing_seeds}))
    df_missing = pd.concat(df_missing)
    
    tmp = pd.concat([tmp,df_missing])
    tmp.fillna(0.0,inplace=True)
    tmp.sort_values(['Tournament','Team'],inplace=True)
    tmp.reset_index(inplace=True, drop=True)
    
    return tmp

In [18]:
#result_df = pd.read_csv('bracket_simulations.csv')
result_df = result_m
result_df['Tournament'] = 'M'
result_df

,Bracket,Slot,Team,Tournament
0,1,R1W1,W01,M
1,1,R1W2,W02,M
2,1,R1W3,W03,M
3,1,R1W4,W04,M
4,1,R1W5,W05,M
...,...,...,...,...
6299995,100000,R4Y1,Y10,M
6299996,100000,R4Z1,Z01,M
6299997,100000,R5WX,W04,M
6299998,100000,R5YZ,Z01,M


In [ ]:
tmp = result_df[['Tournament','Slot','Team']]\
            .groupby(['Tournament','Slot'])\
            .agg('value_counts',normalize=True)

In [ ]:
tmp = tmp.to_frame()
tmp.reset_index(inplace=True)
tmp['Round'] = tmp['Slot'].str[0:2]
tmp.drop(columns='Slot', inplace=True)
tmp.set_index(['Tournament','Team','Round'], inplace=True)
tmp = tmp.stack().unstack(level=2).fillna(0.0)
tmp.reset_index(inplace=True)

In [ ]:
tmp.columns.name=None
tmp.drop(columns='level_2',inplace=True)

# now need to add in missing teams, if any
# some teams may never appear in the bracket.  This means they
# should have implied probabilities of 0 for all rounds
df_missing = []
seeds = [f'{region}{num:02d}' for region in list('WXYZ') \
                for num in range(1,17)]
for t, sdf in tmp.groupby('Tournament'):
    missing_seeds = np.setdiff1d(seeds, sdf['Team'])
    df_missing.append(pd.DataFrame({'Tournament':t, 'Team': missing_seeds}))
df_missing = pd.concat(df_missing)

In [ ]:
tmp = pd.concat([tmp,df_missing])

In [ ]:
tmp.fillna(0.0,inplace=True)
tmp.sort_values(['Tournament','Team'],inplace=True)
tmp.reset_index(inplace=True, drop=True)

In [ ]:
tmp

In [19]:
df_impl_probs = make_implied_probability_table(result_df)

In [20]:
df_impl_probs

,Tournament,Team,R1,R2,R3,R4,R5,R6
0,M,W01,0.97659,0.75393,0.51340,0.35784,0.20221,0.11511
1,M,W02,0.90808,0.67827,0.34896,0.14400,0.05816,0.02211
2,M,W03,0.92889,0.51422,0.30231,0.12360,0.04920,0.01831
3,M,W04,0.85205,0.61202,0.28364,0.16924,0.07839,0.03491
4,M,W05,0.79963,0.30976,0.08921,0.03637,0.01070,0.00303
...,...,...,...,...,...,...,...,...
59,M,Z12,0.27427,0.05807,0.00826,0.00220,0.00032,0.00009
60,M,Z13,0.09648,0.02169,0.00174,0.00034,0.00001,0.00000
61,M,Z14,0.07368,0.01219,0.00088,0.00003,0.00001,0.00000
62,M,Z15,0.08265,0.01342,0.00185,0.00008,0.00001,0.00000


In [21]:
df_impl_probs.to_csv('implied_probs.csv')

In [ ]:
seeds_m